# Screenshot- und Beispielabbildungen

Erzeugt Beispielabbildungen, Metadaten und die dafür verwendeten Frame-Kopien.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
__file__ = str(PROJECT_ROOT / "exports" / "notebooks" / "02_screenshot_beispielabbildungen_notebook.py")
print(f"Projektordner: {PROJECT_ROOT}")


## Code


In [ ]:
from __future__ import annotations

import csv
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps, ImageStat

try:
    import cv2
except Exception:  # pragma: no cover - optional local dependency
    cv2 = None


ROOT = Path(__file__).resolve().parents[2]
FRAME_ROOT = ROOT / "data" / "02_media/stratified_sample/frames"
MERGED_CSV = ROOT / "data" / "04_analysis_results" / "visual_features" / "99_AI_AND_REAL_TIKTOK_VIDEOS_all_results_merged.csv"
HUMAN_AI_CSV = ROOT / "data" / "06_human_evaluation_reannotation" / "human_evaluation_ai_2_annotators.csv"
HUMAN_REAL_CSV = ROOT / "data" / "06_human_evaluation_reannotation" / "human_evaluation_real_2_annotators.csv"

OUT_DIR = ROOT / "exports" / "examples" / "figure_variants"
FINAL_DIR = ROOT / "exports" / "examples" / "figures"
META_DIR = ROOT / "exports" / "examples" / "metadata"
FRAME_OUT_DIR = ROOT / "exports" / "examples" / "frames"
VARIANT_COUNT = 5

TILE_W = 420
TILE_H = 746
GAP = 26
LABEL_H = 170
MARGIN = 40
BG = (248, 247, 242)
PANEL_BG = (255, 255, 255)
TEXT = (28, 31, 35)
MUTED = (78, 84, 93)
VI = (105, 73, 160)
RI = (32, 118, 117)


@dataclass(frozen=True)
class Example:
    video_id: str
    typ: str
    title: str
    subtitle: str
    note: str
    group: str


@dataclass(frozen=True)
class FeatureSpec:
    key: str
    label: str
    eval_col: str
    model_text: Callable[[pd.Series], str]
    diversity_key: Callable[[pd.Series], str]
    interpretation_cols: tuple[str, ...] = ()


def font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        Path("C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf"),
        Path("C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf"),
        Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf"),
        Path("/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size)
    return ImageFont.load_default()


FONT_TITLE = font(21, bold=True)
FONT_SMALL = font(15)
FRAME_CACHE: dict[tuple[str, bool], Path] = {}
FACE_CASCADE = None
if cv2 is not None:
    cascade_path = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
    if cascade_path.exists():
        FACE_CASCADE = cv2.CascadeClassifier(str(cascade_path))


def selected_frame_candidates(video_id: str) -> list[Path]:
    return sorted(FRAME_OUT_DIR.glob(f"*_{video_id}.jpeg"))


def frame_available(video_id: str) -> bool:
    folder = FRAME_ROOT / str(video_id)
    if folder.exists():
        return True
    return bool(selected_frame_candidates(str(video_id)))


def frame_score(path: Path) -> float:
    img = Image.open(path).convert("RGB")
    img.thumbnail((220, 390), Image.Resampling.LANCZOS)
    gray = img.convert("L")
    stat = ImageStat.Stat(gray)
    brightness = stat.mean[0]
    contrast = stat.stddev[0]
    entropy = gray.entropy()
    edges = gray.filter(ImageFilter.FIND_EDGES)
    edge_strength = ImageStat.Stat(edges).mean[0]
    saturation = ImageStat.Stat(img.convert("HSV").split()[1]).mean[0]

    arr = np.asarray(gray)
    very_light = (arr > 242).mean()
    very_dark = (arr < 18).mean()
    brightness_penalty = abs(brightness - 132) / 132
    blank_penalty = very_light + very_dark

    return (
        entropy * 16
        + contrast * 1.4
        + edge_strength * 1.1
        + saturation * 0.08
        - brightness_penalty * 45
        - blank_penalty * 80
    )


def face_score(path: Path) -> float:
    if FACE_CASCADE is None:
        return 0.0
    img = Image.open(path).convert("RGB")
    img.thumbnail((420, 746), Image.Resampling.LANCZOS)
    arr = np.asarray(img)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    faces = FACE_CASCADE.detectMultiScale(gray, scaleFactor=1.08, minNeighbors=4, minSize=(42, 42))
    if len(faces) == 0:
        return 0.0

    h, w = gray.shape
    best = 0.0
    for x, y, fw, fh in faces:
        area_share = (fw * fh) / (w * h)
        center_x = abs((x + fw / 2) - w / 2) / (w / 2)
        center_y = abs((y + fh / 2) - h / 2) / (h / 2)
        centered = max(0.0, 1.0 - (center_x + center_y) / 2)
        best = max(best, 260 * area_share + 55 * centered + 25)
    return best


def best_frame_path(video_id: str, prefer_face: bool = False) -> Path:
    video_id = str(video_id)
    cache_key = (video_id, prefer_face)
    if cache_key in FRAME_CACHE:
        return FRAME_CACHE[cache_key]

    folder = FRAME_ROOT / str(video_id)
    frames = sorted(folder.glob("*.jpeg")) + sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
    if not frames:
        frames = selected_frame_candidates(video_id)
    if not frames:
        raise FileNotFoundError(f"No frames found for {video_id} in {folder} or {FRAME_OUT_DIR}")
    if len(frames) > 18:
        step = max(1, len(frames) // 18)
        candidates = frames[::step][:18]
    else:
        candidates = frames
    if prefer_face:
        best = max(candidates, key=lambda p: frame_score(p) + face_score(p) * 3.0)
    else:
        best = max(candidates, key=frame_score)
    FRAME_CACHE[cache_key] = best
    return best


def fit_image(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return ImageOps.fit(img, (TILE_W, TILE_H), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))


def draw_wrapped(
    draw: ImageDraw.ImageDraw,
    xy: tuple[int, int],
    text: str,
    max_width: int,
    fnt,
    fill=TEXT,
    max_lines: int | None = 2,
) -> int:
    words = str(text).split()
    lines: list[str] = []
    line = ""
    for word in words:
        candidate = f"{line} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=fnt)[2] <= max_width or not line:
            line = candidate
        else:
            lines.append(line)
            line = word
    if line:
        lines.append(line)

    x, y = xy
    line_height = draw.textbbox((0, 0), "Ag", font=fnt)[3] + 6
    visible_lines = lines if max_lines is None else lines[:max_lines]
    for line in visible_lines:
        draw.text((x, y), line, font=fnt, fill=fill)
        y += line_height
    return y


def draw_subtitle(draw: ImageDraw.ImageDraw, xy: tuple[int, int], text: str, max_width: int) -> None:
    x, y = xy
    for raw_line in str(text).splitlines()[:5]:
        y = draw_wrapped(draw, (x, y), raw_line, max_width, FONT_SMALL, fill=MUTED, max_lines=4) + 1


def render_grid(path: Path, examples: list[Example], columns: int) -> None:
    rows = (len(examples) + columns - 1) // columns
    width = MARGIN * 2 + columns * TILE_W + (columns - 1) * GAP
    height = MARGIN * 2 + rows * (TILE_H + LABEL_H) + (rows - 1) * GAP
    canvas = Image.new("RGB", (width, height), BG)
    draw = ImageDraw.Draw(canvas)

    for idx, ex in enumerate(examples):
        row = idx // columns
        col = idx % columns
        x = MARGIN + col * (TILE_W + GAP)
        y = MARGIN + row * (TILE_H + LABEL_H + GAP)

        prefer_face = ex.group in {"beauty_score", "skin_smoothness", "face_emotion", "face_orientation"}
        frame_path = best_frame_path(ex.video_id, prefer_face=prefer_face)
        canvas.paste(fit_image(frame_path), (x, y))

        stripe = VI if ex.typ == "VI" else RI
        draw.rectangle((x, y + TILE_H, x + TILE_W, y + TILE_H + LABEL_H), fill=PANEL_BG)
        draw.rectangle((x, y + TILE_H, x + 9, y + TILE_H + LABEL_H), fill=stripe)
        draw_wrapped(draw, (x + 20, y + TILE_H + 7), ex.title, TILE_W - 32, FONT_TITLE, fill=TEXT)
        draw_subtitle(draw, (x + 20, y + TILE_H + 34), ex.subtitle, TILE_W - 32)

        dst = FRAME_OUT_DIR / f"{path.stem}_{idx + 1:02d}_{ex.typ}_{ex.video_id}.jpeg"
        dst.parent.mkdir(parents=True, exist_ok=True)
        if frame_path.resolve() != dst.resolve():
            shutil.copy2(frame_path, dst)

    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path, quality=95)


def fmt_num(value, digits: int = 2) -> str:
    if pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"


def fmt_int(value) -> str:
    if pd.isna(value):
        return "NA"
    return str(int(round(float(value))))


def clean_label(value) -> str:
    if pd.isna(value):
        return "NA"
    return str(value).replace("_", "-")


def short_label(value) -> str:
    label = clean_label(value)
    replacements = {
        "mediumCloseUp": "medCloseUp",
        "mediumShot": "medShot",
        "extremeLongShot": "extremeLong",
        "beauty-salon": "beauty",
        "airplane-cabin": "airplane",
        "car-interior": "car",
        "forest/broadleaf": "forest",
        "dressing-room": "dressing",
    }
    return replacements.get(label, label)


def clean_interpretation(value) -> str | None:
    if pd.isna(value):
        return None
    label = str(value).strip()
    if not label or label.lower() == "nan":
        return None
    label = re.sub(r"^\d+_", "", label)
    label = label.replace("_", " ")
    replacements = {
        "qualität": "Qualität",
        "farbsattigung": "Farbsättigung",
        "natuerliche": "natürliche",
        "geglaettete": "geglättete",
        "kontrast": "Kontrast",
        "beauty score": "Beauty-Score",
        "haut": "Haut",
        "filterwirkung": "Filterwirkung",
        "schnitte": "Schnitte",
        "bewegung": "Bewegung",
    }
    for old, new in replacements.items():
        label = label.replace(old, new)
    return label


def interpretation_value(row: pd.Series, col: str) -> str | None:
    value = row.get(f"model_interpretation__{col}")
    if pd.isna(value):
        value = row.get(col)
    return clean_interpretation(value)


def interpretation_suffix(row: pd.Series, col: str) -> str:
    label = interpretation_value(row, col)
    return f" ({label})" if label else ""


def rating_to_number(value) -> float | None:
    match = re.match(r"(\d+)_", str(value))
    return float(match.group(1)) if match else None


LIKERT_LABELS = {
    1: "sehr schlecht passend",
    2: "eher schlecht passend",
    3: "teilweise passend",
    4: "gut passend",
    5: "sehr gut passend",
}


def likert_text(value) -> str:
    if pd.isna(value):
        return "keine Angabe"
    score = float(value)
    if score.is_integer() and int(score) in LIKERT_LABELS:
        return LIKERT_LABELS[int(score)]

    lower = int(np.floor(score))
    upper = int(np.ceil(score))
    if lower in LIKERT_LABELS and upper in LIKERT_LABELS and lower != upper:
        return f"zwischen {LIKERT_LABELS[lower]} und {LIKERT_LABELS[upper]}"
    nearest = min(LIKERT_LABELS, key=lambda key: abs(key - score))
    return f"ungefähr {LIKERT_LABELS[nearest]}"


def human_line(row: pd.Series, spec: FeatureSpec) -> str:
    mean_col = f"{spec.eval_col}_mean"
    count_col = f"{spec.eval_col}_count"
    mean_value = row.get(mean_col)
    count_value = row.get(count_col)
    return (
        f"Menschliche Bewertung des Modells: {spec.label}={likert_text(mean_value)} "
        f"({fmt_num(mean_value)}/5 Likert-Skala; n={fmt_int(count_value)})"
    )


def model_line(row: pd.Series, spec: FeatureSpec) -> str:
    return f"Modell: {spec.model_text(row)}"


def concrete_subtitle(row: pd.Series, spec: FeatureSpec) -> str:
    return f"{model_line(row, spec)}\n{human_line(row, spec)}"


def value_bucket(value, cuts: tuple[float, float] = (0.33, 0.66)) -> str:
    if pd.isna(value):
        return "NA"
    val = float(value)
    if val <= cuts[0]:
        return "low"
    if val <= cuts[1]:
        return "mid"
    return "high"


def score_bucket(value, low: float, high: float) -> str:
    if pd.isna(value):
        return "NA"
    val = float(value)
    if val <= low:
        return "low"
    if val >= high:
        return "high"
    return "mid"


def compact_id(video_id: str) -> str:
    return str(video_id)[-6:]


def read_model_data() -> pd.DataFrame:
    df = pd.read_csv(MERGED_CSV)
    df["video_id"] = df["video_id"].astype(str)
    df = df[df["video_id"].map(frame_available)].copy()
    df["typ"] = df["influencer_type"].map({"ai": "VI", "real": "RI"}).fillna(df["influencer_type"])
    return df


def read_human_eval() -> pd.DataFrame:
    parts = [
        pd.read_csv(HUMAN_AI_CSV, sep=None, engine="python"),
        pd.read_csv(HUMAN_REAL_CSV, sep=None, engine="python"),
    ]
    df = pd.concat(parts, ignore_index=True)
    eval_cols = [c for c in df.columns if c.startswith("eval_")]
    interpretation_cols = [c for c in df.columns if c.endswith("_likert_interpretation")]
    numeric = pd.DataFrame({col: df[col].map(rating_to_number) for col in eval_cols})
    keys = df[["video_id", "influencer_type"]].copy()
    keys["video_id"] = keys["video_id"].astype(str)
    numeric = pd.concat([keys, numeric], axis=1)
    means = numeric.groupby(["video_id", "influencer_type"], as_index=False)[eval_cols].mean()
    counts = numeric.groupby(["video_id", "influencer_type"], as_index=False)[eval_cols].count()
    means = means.rename(columns={col: f"{col}_mean" for col in eval_cols})
    counts = counts.rename(columns={col: f"{col}_count" for col in eval_cols})
    human = means.merge(counts, on=["video_id", "influencer_type"], how="left")
    if interpretation_cols:
        interpretations = (
            df[["video_id", "influencer_type", *interpretation_cols]]
            .assign(video_id=lambda frame: frame["video_id"].astype(str))
            .groupby(["video_id", "influencer_type"], as_index=False)
            .agg(lambda series: series.dropna().astype(str).iloc[0] if not series.dropna().empty else np.nan)
            .rename(columns={col: f"model_interpretation__{col}" for col in interpretation_cols})
        )
        human = human.merge(interpretations, on=["video_id", "influencer_type"], how="left")
    human["typ"] = human["influencer_type"].map({"ai": "VI", "real": "RI"})
    return human


def joined_human_model(model_data: pd.DataFrame, human: pd.DataFrame) -> pd.DataFrame:
    joined = human.merge(
        model_data.drop(columns=["influencer_type", "typ"], errors="ignore"),
        on="video_id",
        how="inner",
    )
    joined["typ"] = joined["influencer_type"].map({"ai": "VI", "real": "RI"})
    return joined[joined["video_id"].map(frame_available)].copy()


def pick_ranked_row(rows: pd.DataFrame, rank: int) -> pd.Series:
    if rows.empty:
        raise ValueError("No rows available for ranked selection")
    return rows.iloc[rank % len(rows)]


def pick_ranked_unique(rows: pd.DataFrame, rank: int, used_video_ids: set[str]) -> pd.Series:
    available = rows[~rows["video_id"].astype(str).isin(used_video_ids)]
    if available.empty:
        available = rows
    row = pick_ranked_row(available, rank)
    used_video_ids.add(str(row["video_id"]))
    return row


def visual_quality_for_video(video_id: str) -> float:
    try:
        return frame_score(best_frame_path(str(video_id)))
    except Exception:
        return 0.0


def face_visual_quality_for_video(video_id: str) -> float:
    try:
        frame = best_frame_path(str(video_id), prefer_face=True)
        return frame_score(frame) + face_score(frame) * 4.0
    except Exception:
        return 0.0


def pick_diverse_visual_row(rows: pd.DataFrame, spec: FeatureSpec, rank: int, used_video_ids: set[str], used_diversity: set[str]) -> pd.Series:
    available = rows[~rows["video_id"].astype(str).isin(used_video_ids)].copy()
    if available.empty:
        available = rows.copy()

    available["_diversity"] = available.apply(spec.diversity_key, axis=1).astype(str)
    diverse = available[~available["_diversity"].isin(used_diversity)]
    if diverse.empty:
        diverse = available

    top_n = min(len(diverse), max(8, rank + 4))
    candidates = diverse.head(top_n).copy()
    if spec.key in {"beauty_score", "skin_smoothness", "face_emotion", "face_orientation"}:
        candidates["_visual_quality"] = candidates["video_id"].map(face_visual_quality_for_video)
    else:
        candidates["_visual_quality"] = candidates["video_id"].map(visual_quality_for_video)
    candidates = candidates.sort_values(["_visual_quality", "video_id"], ascending=[False, True])

    row = candidates.iloc[rank % len(candidates)]
    used_video_ids.add(str(row["video_id"]))
    used_diversity.add(str(row["_diversity"]))
    return row.drop(labels=[c for c in ["_diversity", "_visual_quality"] if c in row.index])


def scene_text(row: pd.Series) -> str:
    return (
        f"scene={short_label(row.get('scene_classification__scene_top1_label'))}; "
        f"p={fmt_num(row.get('scene_classification__scene_top1_confidence'))}; "
        f"scene_n={fmt_int(row.get('scene_classification__scene_unique_labels'))}"
    )


def feature_specs() -> list[FeatureSpec]:
    return [
        FeatureSpec(
            "scene",
            "Szene",
            "eval_scene_context",
            scene_text,
            lambda r: short_label(r.get("scene_classification__scene_top1_label")),
        ),
        FeatureSpec(
            "camera_distance",
            "Kameradistanz",
            "eval_camera_distance",
            lambda r: (
                f"distance={short_label(r.get('camera_distance__camera_distance_label'))}; "
                f"p={fmt_num(r.get('camera_distance__camera_distance_confidence'))}; "
                f"frames={fmt_int(r.get('camera_distance__processed_frame_count'))}"
            ),
            lambda r: short_label(r.get("camera_distance__camera_distance_label")),
        ),
        FeatureSpec(
            "camera_stability",
            "Stabilität",
            "eval_camera_stability",
            lambda r: (
                f"stability_idx={fmt_num(r.get('camera_stability__stability_index'))}"
                f"{interpretation_suffix(r, 'camera_stability__stability_likert_interpretation')}; "
                f"flow_mean={fmt_num(r.get('camera_stability__optical_flow_magnitude_mean'))}"
                f"{interpretation_suffix(r, 'camera_stability__optical_flow_likert_interpretation')}; "
                f"pairs={fmt_int(r.get('camera_stability__processed_frame_pairs'))}"
            ),
            lambda r: score_bucket(r.get("camera_stability__stability_index"), 0.45, 0.75),
            (
                "camera_stability__stability_likert_interpretation",
                "camera_stability__optical_flow_likert_interpretation",
            ),
        ),
        FeatureSpec(
            "cuts",
            "Schnitte/Dynamik",
            "eval_cuts_dynamik",
            lambda r: (
                f"cut_count={fmt_int(r.get('cuts__cut_count'))}"
                f"{interpretation_suffix(r, 'cuts__cut_count_likert_interpretation')}; "
                f"cuts_s={fmt_num(r.get('cuts__cuts_per_second'))}"
                f"{interpretation_suffix(r, 'cuts__cuts_per_second_likert_interpretation')}; "
                f"frames={fmt_int(r.get('cuts__frames_scanned'))}"
            ),
            lambda r: score_bucket(r.get("cuts__cuts_per_second"), 0.05, 0.50),
            (
                "cuts__cut_count_likert_interpretation",
                "cuts__cuts_per_second_likert_interpretation",
            ),
        ),
        FeatureSpec(
            "face_emotion",
            "Emotion",
            "eval_face_emotion",
            lambda r: (
                f"emotion={short_label(r.get('face_emotion__emotion_major_beit_readable'))}; "
                f"p={fmt_num(r.get('face_emotion__emotion_major_beit_confidence'))}; "
                f"face_frames={fmt_int(r.get('face_emotion__detected_emotion_frames'))}"
            ),
            lambda r: short_label(r.get("face_emotion__emotion_major_beit_readable")),
        ),
        FeatureSpec(
            "body_pose",
            "Pose",
            "eval_body_pose",
            lambda r: (
                f"pose={short_label(r.get('body_pose__pose_orientation'))}; "
                f"p={fmt_num(r.get('body_pose__pose_confidence'))}; "
                f"pose_frames={fmt_int(r.get('body_pose__detected_pose_frames'))}"
            ),
            lambda r: short_label(r.get("body_pose__pose_orientation")),
        ),
        FeatureSpec(
            "beauty_score",
            "Beauty-Score",
            "eval_beauty_score",
            lambda r: (
                f"beauty_mean={fmt_num(r.get('beauty_scoring__beauty_score_mean'))}"
                f"{interpretation_suffix(r, 'beauty_scoring__beauty_score_likert_interpretation')}; "
                f"sd={fmt_num(r.get('beauty_scoring__beauty_score_std'))}; "
                f"face_frames={fmt_int(r.get('beauty_scoring__beauty_detected_face_frames'))}"
            ),
            lambda r: score_bucket(r.get("beauty_scoring__beauty_score_mean"), 25, 40),
            ("beauty_scoring__beauty_score_likert_interpretation",),
        ),
        FeatureSpec(
            "skin_smoothness",
            "Hautglättung",
            "eval_skin_smoothness",
            lambda r: (
                f"highpass_idx={fmt_num(r.get('skin_smoothness__skin_smoothness_highpass_index'))}"
                f"{interpretation_suffix(r, 'skin_smoothness__highpass_likert_interpretation')}; "
                f"dog_idx={fmt_num(r.get('skin_smoothness__skin_smoothness_dog_index'))}"
                f"{interpretation_suffix(r, 'skin_smoothness__dog_likert_interpretation')}; "
                f"face_frames={fmt_int(r.get('skin_smoothness__detected_skin_face_frames'))}"
            ),
            lambda r: score_bucket(r.get("skin_smoothness__skin_smoothness_highpass_index"), 0.03, 0.08),
            (
                "skin_smoothness__highpass_likert_interpretation",
                "skin_smoothness__dog_likert_interpretation",
            ),
        ),
        FeatureSpec(
            "face_orientation",
            "Gesichtsorientierung",
            "eval_face_orientation",
            lambda r: (
                f"orientation={short_label(r.get('angle_face_orientation__angle_face_orientation'))}; "
                f"pitch={fmt_num(r.get('angle_face_orientation__face_pitch_mean'))}; "
                f"yaw={fmt_num(r.get('angle_face_orientation__face_yaw_mean'))}"
            ),
            lambda r: short_label(r.get("angle_face_orientation__angle_face_orientation")),
        ),
        FeatureSpec(
            "brightness",
            "Helligkeit",
            "eval_brightness",
            lambda r: (
                f"brightness_idx={fmt_num(r.get('brightness_contrast__brightness_index'))}"
                f"{interpretation_suffix(r, 'brightness_contrast__brightness_likert_interpretation')}"
            ),
            lambda r: short_label(r.get("brightness_contrast__brightness_likert_interpretation")),
            ("brightness_contrast__brightness_likert_interpretation",),
        ),
        FeatureSpec(
            "contrast",
            "Kontrast",
            "eval_contrast",
            lambda r: (
                f"contrast_idx={fmt_num(r.get('brightness_contrast__contrast_index'))}"
                f"{interpretation_suffix(r, 'brightness_contrast__contrast_likert_interpretation')}"
            ),
            lambda r: short_label(r.get("brightness_contrast__contrast_likert_interpretation")),
            ("brightness_contrast__contrast_likert_interpretation",),
        ),
        FeatureSpec(
            "saturation",
            "Sättigung",
            "eval_saturation",
            lambda r: (
                f"saturation_idx={fmt_num(r.get('saturation__saturation_index'))}"
                f"{interpretation_suffix(r, 'saturation__saturation_likert_interpretation')}"
            ),
            lambda r: short_label(r.get("saturation__saturation_likert_interpretation")),
            ("saturation__saturation_likert_interpretation",),
        ),
        FeatureSpec(
            "sharpness",
            "Schärfe",
            "eval_sharpness",
            lambda r: (
                f"laplacian_mean={fmt_num(r.get('visual_sharpness__sharpness_laplacian_mean'))}"
                f"{interpretation_suffix(r, 'visual_sharpness__sharpness_likert_interpretation')}"
            ),
            lambda r: short_label(r.get("visual_sharpness__sharpness_likert_interpretation")),
            ("visual_sharpness__sharpness_likert_interpretation",),
        ),
        FeatureSpec(
            "aesthetic_quality",
            "Ästhetik",
            "eval_aesthetic_quality",
            lambda r: (
                f"aesthetic_score={fmt_num(r.get('aesthetic_quality__aesthetic_quality_score'))}"
                f"{interpretation_suffix(r, 'aesthetic_quality__score_likert_interpretation')}; "
                f"frames={fmt_int(r.get('aesthetic_quality__aesthetic_quality_scored_frames'))}"
            ),
            lambda r: score_bucket(r.get("aesthetic_quality__aesthetic_quality_score"), 3.0, 4.0),
            ("aesthetic_quality__score_likert_interpretation",),
        ),
        FeatureSpec(
            "filter_strength",
            "Filterstärke",
            "eval_filter_strength",
            lambda r: (
                f"filter={short_label(r.get('visual_filter__filter_strength_label'))}"
                f"{interpretation_suffix(r, 'visual_filter__filter_strength_likert_interpretation')}; "
                f"prob={fmt_num(r.get('visual_filter__filter_strength_prob'))}; "
                f"frames={fmt_int(r.get('visual_filter__processed_frame_count'))}"
            ),
            lambda r: short_label(r.get("visual_filter__filter_strength_label")),
            ("visual_filter__filter_strength_likert_interpretation",),
        ),
        FeatureSpec(
            "personen_anzahl",
            "Personenzahl",
            "eval_personen_anzahl",
            lambda r: (
                f"persons={fmt_num(r.get('structural_personen_anzahl__personen_anzahl'))}; "
                f"max={fmt_int(r.get('structural_personen_anzahl__personen_anzahl_max'))}; "
                f"frames={fmt_int(r.get('structural_personen_anzahl__detected_person_frames'))}"
            ),
            lambda r: score_bucket(r.get("structural_personen_anzahl__personen_anzahl"), 1.2, 2.2),
        ),
    ]


def pick_scene(model_data: pd.DataFrame, typ: str, label: str, title: str, note: str, rank: int = 0) -> Example:
    rows = model_data[
        (model_data["typ"] == typ) & (model_data["scene_classification__scene_top1_label"] == label)
    ].sort_values("scene_classification__scene_top1_confidence", ascending=False)
    if rows.empty:
        rows = model_data[model_data["typ"] == typ].sort_values(
            ["scene_classification__scene_top1_confidence", "video_id"],
            ascending=[False, True],
        )
    if rows.empty:
        raise ValueError(f"No example found for {typ}/{label}")
    row = pick_ranked_row(rows, rank)
    subtitle = (
        f"Modell: {scene_text(row)}\n"
        f"Modell: stability_idx={fmt_num(row.get('camera_stability__stability_index'))}; "
        f"cuts_s={fmt_num(row.get('cuts__cuts_per_second'))}\n"
        f"Modell: cam={short_label(row.get('camera_distance__camera_distance_label'))}; "
        f"p={fmt_num(row.get('camera_distance__camera_distance_confidence'))}"
    )
    title_with_id = f"{title} | {typ} | id={compact_id(row['video_id'])}"
    return Example(row["video_id"], typ, title_with_id, subtitle, note, "scene")


def pick_feature_examples(joined: pd.DataFrame, spec: FeatureSpec, variant: int) -> list[Example]:
    rating_col = f"{spec.eval_col}_mean"
    rows = joined.dropna(subset=[rating_col]).copy()
    rows = rows[rows["video_id"].map(frame_available)]
    if spec.key == "beauty_score":
        rows = rows[
            (rows["beauty_scoring__beauty_detected_face_frames"].fillna(0) >= 3)
            & rows["camera_distance__camera_distance_label"].isin(["closeUp", "mediumCloseUp", "mediumShot"])
        ]
    if spec.key == "skin_smoothness":
        rows = rows[
            (rows["skin_smoothness__detected_skin_face_frames"].fillna(0) >= 3)
            & rows["camera_distance__camera_distance_label"].isin(["closeUp", "mediumCloseUp", "mediumShot"])
        ]
    if rows.empty:
        return []

    high = rows.sort_values([rating_col, "video_id"], ascending=[False, True])
    low = rows.sort_values([rating_col, "video_id"], ascending=[True, True])
    median_value = rows[rating_col].median()
    mid = rows.assign(_mid_dist=(rows[rating_col] - median_value).abs()).sort_values(["_mid_dist", "video_id"])

    used: set[str] = set()
    used_diversity: set[str] = set()
    selections = [
        pick_diverse_visual_row(high, spec, variant, used, used_diversity),
        pick_diverse_visual_row(mid, spec, variant, used, used_diversity),
        pick_diverse_visual_row(low, spec, variant, used, used_diversity),
    ]

    examples = []
    for idx, row in enumerate(selections, start=1):
        examples.append(
            Example(
                video_id=str(row["video_id"]),
                typ=row["typ"],
                title=f"{spec.label} | {row['typ']} | id={compact_id(row['video_id'])}",
                subtitle=concrete_subtitle(row, spec),
                note=f"{spec.key}_variant_{variant + 1}",
                group=spec.key,
            )
        )
    return examples


def make_human_overview_examples(joined: pd.DataFrame, specs_by_key: dict[str, FeatureSpec], variant: int) -> list[Example]:
    overview_keys = [
        ["scene", "camera_distance", "body_pose"],
        ["face_emotion", "beauty_score", "skin_smoothness"],
        ["camera_stability", "cuts", "personen_anzahl"],
    ][variant % 3]

    examples = []
    for idx, key in enumerate(overview_keys, start=1):
        spec = specs_by_key[key]
        feature_examples = pick_feature_examples(joined, spec, variant)
        if feature_examples:
            ex = feature_examples[min(idx - 1, len(feature_examples) - 1)]
            examples.append(
                Example(
                    video_id=ex.video_id,
                    typ=ex.typ,
                    title=ex.title,
                    subtitle=ex.subtitle,
                    note=ex.note,
                    group="human_evaluation_overview",
                )
            )
    return examples


def write_metadata(figures: dict[str, list[Example]]) -> None:
    META_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = META_DIR / "screenshot_examples_metadata.csv"
    with csv_path.open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=["figure", "position", "video_id", "type", "title", "subtitle", "note", "group"])
        writer.writeheader()
        for figure, examples in figures.items():
            for pos, ex in enumerate(examples, start=1):
                writer.writerow(
                    {
                        "figure": figure,
                        "position": pos,
                        "video_id": ex.video_id,
                        "type": ex.typ,
                        "title": ex.title,
                        "subtitle": ex.subtitle,
                        "note": ex.note,
                        "group": ex.group,
                    }
                )

def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    META_DIR.mkdir(parents=True, exist_ok=True)
    FRAME_OUT_DIR.mkdir(parents=True, exist_ok=True)
    # Manuell kuratierte Abbildungen wie abb_influencerinnen_auswahl.png bleiben erhalten.
    managed_patterns = [
        "abb_datasetbeispiele_vi_ri*.png",
        "abb_human_evaluation_modellplausibilitaet*.png",
        "abb_visuelle_settings_szenen*.png",
        "abb_kategorie_*.png",
    ]
    for pattern in managed_patterns:
        for old_png in OUT_DIR.glob(pattern):
            old_png.unlink()
    model_data = read_model_data()
    human = read_human_eval()
    joined = joined_human_model(model_data, human)
    specs = feature_specs()
    specs_by_key = {spec.key: spec for spec in specs}

    figures: dict[str, list[Example]] = {}
    for variant in range(VARIANT_COUNT):
        suffix = f"_v{variant + 1:02d}"

        dataset_examples = [
            pick_scene(model_data, "VI", "beauty_salon", "VI 1", "Datensatzbeispiel", variant),
            pick_scene(model_data, "VI", "stage/indoor", "VI 2", "Datensatzbeispiel", variant),
            pick_scene(model_data, "VI", "playroom", "VI 3", "Datensatzbeispiel", variant),
            pick_scene(model_data, "RI", "beauty_salon", "RI 1", "Datensatzbeispiel", variant),
            pick_scene(model_data, "RI", "airplane_cabin", "RI 2", "Datensatzbeispiel", variant),
            pick_scene(model_data, "RI", "park", "RI 3", "Datensatzbeispiel", variant),
        ]
        human_examples = make_human_overview_examples(joined, specs_by_key, variant)
        setting_examples = [
            pick_scene(model_data, "VI", "stage/indoor", "VI | stage/indoor", "Szenenbeispiel", variant),
            pick_scene(model_data, "VI", "playroom", "VI | playroom", "Szenenbeispiel", variant),
            pick_scene(model_data, "VI", "beauty_salon", "VI | beauty_salon", "Szenenbeispiel", variant),
            pick_scene(model_data, "RI", "airplane_cabin", "RI | airplane_cabin", "Szenenbeispiel", variant),
            pick_scene(model_data, "RI", "car_interior", "RI | car_interior", "Szenenbeispiel", variant),
            pick_scene(model_data, "RI", "park", "RI | park", "Szenenbeispiel", variant),
        ]

        variant_figures = {
            f"abb_datasetbeispiele_vi_ri{suffix}.png": dataset_examples,
            f"abb_human_evaluation_modellplausibilitaet{suffix}.png": human_examples,
            f"abb_visuelle_settings_szenen{suffix}.png": setting_examples,
        }

        for spec in specs:
            examples = pick_feature_examples(joined, spec, variant)
            if examples:
                variant_figures[f"abb_kategorie_{spec.key}{suffix}.png"] = examples

        for filename, examples in variant_figures.items():
            render_grid(OUT_DIR / filename, examples, columns=3)
        figures.update(variant_figures)

        if variant == 0:
            aliases = {
                "abb_datasetbeispiele_vi_ri.png": dataset_examples,
                "abb_human_evaluation_modellplausibilitaet.png": human_examples,
                "abb_visuelle_settings_szenen.png": setting_examples,
            }
            for filename, examples in aliases.items():
                render_grid(FINAL_DIR / filename, examples, columns=3)
            figures.update(aliases)

    write_metadata(figures)
    print(f"Created {len(figures)} figures in {OUT_DIR}")
    print(f"Copied selected source frames to {FRAME_OUT_DIR}")



## Ausführung


In [ ]:
main()
